In [1]:
pip install anthropic


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
from dotenv import load_dotenv
load_dotenv(override=True)  # force reload even if key was already set in this session
import os
import chromadb
from sentence_transformers import SentenceTransformer
import anthropic

# ── 1. Load the same embedding model used during storage ─────────────────────
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# ── 2. Connect to existing ChromaDB ──────────────────────────────────────────
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="cricket_docs")

print(f"ChromaDB loaded — {collection.count()} chunks available\n")

# ── 3. Connect to Claude (reads ANTHROPIC_API_KEY from .env automatically) ───
claude = anthropic.Anthropic()

# ── 4. Retrieval function ─────────────────────────────────────────────────────
def retrieve_chunks(question, top_k=5):
    """
    Embeds the question and finds the top_k most similar chunks in ChromaDB
    """
    question_embedding = model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k
    )

    chunks = results['documents'][0]
    sources = [m['source'] for m in results['metadatas'][0]]

    return chunks, sources

# ── 5. RAG function ───────────────────────────────────────────────────────────
def ask_cricket_bot(question):
    """
    Retrieves relevant chunks and sends them to Claude to generate an answer
    """
    print(f"Question: {question}")
    print("-" * 50)

    chunks, sources = retrieve_chunks(question)

    context = "\n\n".join([f"[Source: {src}]\n{chunk}" 
                           for src, chunk in zip(sources, chunks)])

    prompt = f"""You are a cricket expert assistant. Answer the question using ONLY 
the context provided below. If the answer is not in the context, say 
"I don't have enough information about that."

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

    response = claude.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    answer = response.content[0].text

    print(f"Answer: {answer}")
    print(f"\nSources used: {list(set(sources))}")
    print("=" * 50)

    return answer, sources

# ── 6. Test it out ────────────────────────────────────────────────────────────
ask_cricket_bot("How many international centuries did Sachin Tendulkar score?")
ask_cricket_bot("What is LBW in cricket?")
ask_cricket_bot("How does the IPL work?")

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6128.72it/s]


ChromaDB loaded — 496 chunks available

Question: How many international centuries did Sachin Tendulkar score?
--------------------------------------------------
Answer: According to the context provided, Sachin Tendulkar scored **100 international centuries** combined across Tests and ODIs. 

Specifically, the context mentions:
- He holds the record for the highest number of centuries in Tests: **51 centuries**
- He stands second in ODIs with **49 centuries** (behind Virat Kohli)
- He has the most number of centuries when Tests and ODIs are combined: **100 centuries**

The context also notes that he was "the only batsman to score 100 international centuries."

Sources used: ['Sachin_Tendulkar.txt']
Question: What is LBW in cricket?
--------------------------------------------------
Answer: Based on the context provided, LBW stands for "Leg before wicket, a way in which a batter can be dismissed in cricket."

Sources used: ['Cricket.txt', 'LBW.txt', 'Cricket_ball.txt', 'Bowling_(cricke

("Based on the context provided, here's how the IPL works:\n\n**Format and Structure:**\n- The IPL is a professional Twenty20 (T20) cricket league organized by the Board of Control for Cricket in India (BCCI)\n- It features ten city-based franchise teams\n- The tournament is held annually between March and May\n\n**Team Ownership:**\n- Franchises are owned through an auction system. The first franchise auction was held on 24 January 2008, where eight franchises were sold for a total of $723.59 million\n\n**Player Contracts:**\n- Players are contracted only for the duration of the tournament, which is less than two months\n- Players are acquired through a player auction system\n- Since the 2014 season, both domestic and international players are included in the player auction\n- Overseas players can be remunerated in their currency of choice\n- The BCCI gives 10% of foreign players' salaries to their country's national cricket board\n\n**Competition Structure:**\n- Teams compete in the 

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

key = os.getenv("ANTHROPIC_API_KEY")
print(f"Key found: {key is not None}")
print(f"Key starts with: {key[:10] if key else 'NONE'}")

Key found: True
Key starts with: sk-ant-api
